In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import VarianceThreshold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Wczytaj dane
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Parametry
thresholds = np.arange(0.0, 0.2, 0.01)
k_values = [1, 3, 5, 7, 9]

best_score = 0
best_k = None
best_threshold = None
best_features = None
best_X_reduced = None

# Szukanie najlepszego zestawu
for t in thresholds:
    selector = VarianceThreshold(threshold=t)
    X_reduced = selector.fit_transform(X_scaled)
    selected_mask = selector.get_support()
    selected_feature_names = feature_names[selected_mask]

    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        scores = cross_val_score(knn, X_reduced, y, cv=5, scoring='accuracy')
        mean_score = scores.mean()

        if mean_score > best_score:
            best_score = mean_score
            best_k = k
            best_threshold = t
            best_features = selected_feature_names
            best_X_reduced = X_reduced.copy()

# Zapis .npy
np.save("best_X_knn_reduced.npy", best_X_reduced)

# Zapis .txt
with open("best_features_knn.txt", "w") as f:
    f.write(f"Najlepszy próg wariancji: {best_threshold:.3f}\n")
    f.write(f"Najlepsze k: {best_k}\n")
    f.write(f"Dokładność (accuracy): {best_score:.4f}\n")
    f.write("Wybrane cechy:\n")
    for feat in best_features:
        f.write(f"- {feat}\n")

# Zapis .csv
df_reduced = pd.DataFrame(best_X_reduced, columns=best_features)
df_reduced['target'] = y  # dodaj etykiety klas
df_reduced.to_csv("best_X_knn_reduced.csv", index=False)

print("Zapisano pliki:")
print("- best_X_knn_reduced.npy")
print("- best_features_knn.txt")
print("- best_X_knn_reduced.csv")


✅ Zapisano pliki:
- best_X_knn_reduced.npy
- best_features_knn.txt
- best_X_knn_reduced.csv
